# Practice 03: Control and linearization with MuJoCo

**Goal:** connect actuator models, feedback laws, and local numerical derivatives on a Menagerie UR5e.

**Learning path**

1. Load and render the robot in a fresh kernel.
2. Understand actuator force laws and joint-space actuation maps.
3. Compare an internal position servo with external PD torque control.
4. Estimate gravity sensitivities and a discrete-time state linearization.
5. Check the linearization against an actual perturbed step.

This notebook is independent of p01; no variables or renderer are shared. For robot modeling, floating bases, and contacts, see [practice 01](../p01-simulation/tutorial.ipynb).

## Setup

From the repository root:

```bash
uv sync --locked --extra p03
uv run --locked --extra p03 jupyter lab practices/p03-manipulator/control_and_linearization.ipynb
```

With the course conda environment activated:

```bash
python scripts/setup_conda.py p03
python -m jupyterlab practices/p03-manipulator/control_and_linearization.ipynb
```

The [setup guide](../README.md) covers kernels, graphics, and FFmpeg. Choose a headless graphics backend before importing MuJoCo if necessary. The UR5e downloads on first use and is then cached.

### Import the tools

MuJoCo provides physics, Menagerie supplies the robot, NumPy handles arrays, and MediaPy/Matplotlib display the results.

In [ ]:
import mujoco
import mujoco_menagerie as mm
import mediapy as media
import numpy as np
import matplotlib.pyplot as plt

renderer = None
rng = np.random.default_rng(0)
print("MuJoCo:", mujoco.__version__)

## 1. Load and view the UR5e

Load a fresh scene, select its supplied initial keyframe, and render it before running calculations. Set the physics timestep to 1 ms. The graphics resources belong to this model and are closed at the end.

In [ ]:
model = mm.load("universal_robots_ur5e")
data = mujoco.MjData(model)
mujoco.mj_forward(model, data)
print(f"nq={model.nq}, nv={model.nv}, nu={model.nu}")
model.opt.timestep = 0.001
model.opt.integrator = mujoco.mjtIntegrator.mjINT_IMPLICIT
if renderer is not None:
    renderer.close()
if model.nkey:
    mujoco.mj_resetDataKeyframe(model, data, 0)
mujoco.mj_forward(model, data)

model.vis.global_.offwidth = 800
model.vis.global_.offheight = 600
renderer = mujoco.Renderer(model, width=800, height=600)
preview_camera = mujoco.MjvCamera()
preview_camera.lookat[:] = model.stat.center
preview_camera.distance = 1.8 * model.stat.extent
preview_camera.azimuth = 135
preview_camera.elevation = -20
renderer.update_scene(data, camera=preview_camera)
robot_preview = renderer.render()
media.show_image(robot_preview)
print("UR5e scene loaded; simulated time:", data.time, "s")
q_nominal = np.array([-0.6, -1.2, -2.1, 0.7, 0.7, 0.7])

## 2. Actuator models

### Actuator force laws

A stateless actuator with fixed gain and affine bias produces

$$
p=a u+b_0+b_1l+b_2\dot l.
$$

For this notebook's direct unit-gear joint transmissions, actuator length and velocity correspond to joint angle and angular velocity.

**The three configurations are:**

$$
\begin{aligned}
\text{Motor:}\quad &p=u,\\
\text{Position servo:}\quad &p=k_p(u-q)-k_dv,\\
\text{Velocity servo:}\quad &p=k_v(u-v).
\end{aligned}
$$

**Next cell:** define a helper that sets both the actuator types and their numerical parameters.

Control and force limits remain active. A saturated actuator may deliver less torque than requested.

In [ ]:
class ActuatorMotor:
    def __init__(self, torque_range = [-100,100]) -> None:
        self.range = torque_range
        self.dyn = np.array([1., 0., 0.])
        self.gain = np.array([1., 0., 0.])
        self.bias = np.array([0., 0., 0.])

    def __repr__(self) -> str:
        return f"ActuatorMotor(dyn={self.dyn}, gain={self.gain}, bias={self.bias})"


class ActuatorPosition(ActuatorMotor):
    def __init__(self, kp=1, kd=0, position_range = [-100,100]) -> None:
        super().__init__()
        self.range = position_range
        self.kp = kp
        self.kd = kd
        self.gain[0] = self.kp
        self.bias[1] = -self.kp
        self.bias[2] = -self.kd


class ActuatorVelocity(ActuatorMotor):
    def __init__(self, kv=1,  velocity_range = [-100,100]) -> None:
        super().__init__()
        self.range = velocity_range
        self.kv = kv
        self.gain[0] = self.kv
        self.bias[2] = -self.kv

def update_actuator(model, actuator_id, actuator):
    """
    Update actuator in model
    model - mujoco.MjModel
    actuator_id - int or str (name) (for reference see, named access to model elements)
    actuator - ActuatorMotor, ActuatorPosition, ActuatorVelocity
    """
    actuator_id = model.actuator(actuator_id).id
    model.actuator(actuator_id).dynprm = np.zeros(len(model.actuator(actuator_id).dynprm))
    model.actuator(actuator_id).gainprm = np.zeros(len(model.actuator(actuator_id).gainprm))
    model.actuator(actuator_id).biasprm = np.zeros(len(model.actuator(actuator_id).biasprm))
    model.actuator(actuator_id).ctrlrange = actuator.range 
    model.actuator(actuator_id).dynprm[:3] = actuator.dyn
    model.actuator(actuator_id).gainprm[:3] = actuator.gain
    model.actuator(actuator_id).biasprm[:3] = actuator.bias
    model.actuator_dyntype[actuator_id] = mujoco.mjtDyn.mjDYN_NONE
    model.actuator_gaintype[actuator_id] = mujoco.mjtGain.mjGAIN_FIXED
    model.actuator_biastype[actuator_id] = mujoco.mjtBias.mjBIAS_AFFINE
    model.actuator_ctrllimited[actuator_id] = True

### Direct-joint actuation map

For unit-gain motors on direct hinge transmissions:

$$
\tau_{\mathrm{actuator}}=Su.
$$

Construct the selector from joint **velocity addresses** and transmission gear ratios rather than assuming actuator order matches joint order. This six-joint UR5e is fully actuated; a legged robot driven only by joint motors has six unactuated base coordinates, as discussed in p01.

**Next cell:** switch to torque motors, construct the selector, and compare its prediction with the delivered generalized force. The chosen test command is below all control and force limits.

In [ ]:
for actuator_id in range(model.nu):
    update_actuator(model, actuator_id, ActuatorMotor())
S = np.zeros((model.nv, model.nu))
for actuator_id in range(model.nu):
    assert model.actuator_trntype[actuator_id] == mujoco.mjtTrn.mjTRN_JOINT
    joint_id = model.actuator_trnid[actuator_id, 0]
    assert model.jnt_type[joint_id] == mujoco.mjtJoint.mjJNT_HINGE
    S[model.jnt_dofadr[joint_id], actuator_id] = model.actuator_gear[actuator_id, 0]
map_data = mujoco.MjData(model)
map_data.qpos[:] = q_nominal
map_data.ctrl[:] = np.linspace(-0.4, 0.4, model.nu)
mujoco.mj_forward(model, map_data)
np.testing.assert_allclose(map_data.qfrc_actuator, S @ map_data.ctrl, atol=1e-12)
print("Actuation map shape / rank:", S.shape, np.linalg.matrix_rank(S))

## 3. Track a target with a position servo

The servo computes proportional and derivative feedback internally; `data.ctrl` contains target angles.

**Simulation loop:**

1. Reset the state.
2. Assign the target before each physics step.
3. Step the simulation.
4. Save copies of joint positions and sample video frames.

Copies are essential because MuJoCo's arrays are mutable. Without gravity feedforward, a steady-state position error can remain.

In [ ]:
# update actuators
position_motor = ActuatorPosition(kp=200, kd=100)

for actuator_id in range(model.nu):
    update_actuator(model, actuator_id, position_motor)

# setup simulation
q_des = np.array([-0.6,-1.2,-2.1,0.7,0.7,0.7])

duration = 5  # (seconds)
framerate = 30  # (Hz)
n_steps = int(np.ceil(duration * framerate)) + 1

# Simulate and display video.
frames = []
mujoco.mj_resetData(model, data)  # Reset state and time.

history_q = []

while data.time < duration:
    data.ctrl[:] = q_des

    mujoco.mj_step(model, data)

    history_q.append(data.qpos.copy())

    if len(frames) < data.time * framerate:
        mujoco.mj_forward(model, data)
        renderer.update_scene(data)
        pixels = renderer.render()
        frames.append(pixels)

media.show_video(frames, fps=framerate)
assert np.isfinite(data.qpos).all() and np.isfinite(data.qvel).all()
assert np.isfinite(history_q).all()


### Read the response plot

- **Horizontal axis:** simulated time, in seconds.
- **Vertical axis:** joint angle, in radians.
- **Solid curves:** measured joint angles.
- **Dashed lines:** target angles.

Compare transient motion, settling, and final tracking error.

In [ ]:
import matplotlib.pyplot as plt

history_q = np.array(history_q)

plt.hlines(q_des, 0, len(history_q) * model.opt.timestep, linestyles="dashed")
plt.plot(np.arange(len(history_q)) * model.opt.timestep, history_q, label=["q" + str(i) for i in range(6)])
plt.xlabel("Time [s]")
plt.ylabel("Joint angle [rad]")
plt.grid()
plt.legend()
plt.show()

### Compute PD torque externally

Switch to unit-gain motors so the control input requests torque in **N·m**:

$$
u=K_p(q_d-q)-K_dv.
$$

- Proportional feedback pulls the joint toward its target.
- Derivative feedback damps motion.
- Compute the command **before** calling `mj_step`.

Gravity and actuator saturation make the ideal linear error dynamics only an approximation.

In [ ]:
# mujoco.mj_reset(model, data)

# setup simulation
q0 = np.array([-3.6,-1.2,-2.1,0.7,0.7,0.7])
q_des = np.array([-2.6,-0.8,-1.9,0.7,0.7,0.7])


mujoco.mj_resetData(model, data)  # Reset state and time.
data.qpos = q0
data.qvel = np.zeros(model.nv)
# set up actuators
torque_motor = ActuatorMotor()

for actuator_id in range(model.nu):
    update_actuator(model, actuator_id, torque_motor)

duration = 2.3  # (seconds)
framerate = 60  # (Hz)

# Simulate and display video.
frames = []
history_q = []
while data.time < duration:
    # get current joint positions, velocities
    q = data.qpos.copy()
    dq = data.qvel.copy()
    history_q.append(q)

    # implement PD control
    data.ctrl[:] = 250.*(q_des - q) - 50.*dq #np.zeros(model.nu)
    mujoco.mj_step(model, data)
    
    if len(frames) < data.time * framerate:
        mujoco.mj_forward(model, data)
        renderer.update_scene(data)
        pixels = renderer.render()
        frames.append(pixels)
        
media.show_video(frames, fps=framerate)
assert np.isfinite(data.qpos).all() and np.isfinite(data.qvel).all()
assert np.isfinite(history_q).all()


### Compare the two controllers

Use the trajectories and targets to compare:

- How quickly the joints settle.
- Whether they overshoot.
- The final tracking error.

The gains and actuator saturation affect both responses.

In [ ]:
history_q = np.array(history_q)

plt.hlines(q_des, 0, len(history_q) * model.opt.timestep, linestyles="dashed")
plt.plot(np.arange(len(history_q)) * model.opt.timestep, history_q, label=["q" + str(i) for i in range( 6)])
plt.xlabel("Time [s]")
plt.ylabel("Joint angle [rad]")
plt.grid()
plt.legend()
plt.show()

## 4. Gravity sensitivities

MuJoCo evaluates numerical dynamics quantities. A local gravity derivative can be estimated with central differences:

$$
\frac{\partial g}{\partial q_j}\approx
\frac{g(q+\varepsilon e_j)-g(q-\varepsilon e_j)}{2\varepsilon}.
$$

- Evaluate `qfrc_bias` at zero velocity to obtain gravity.
- Use separate data objects to preserve the current simulation state.
- Perturb configuration with `mj_integratePos` rather than assuming all models use Euclidean coordinates.
- Halve the perturbation to check consistency; extremely small steps amplify round-off error.

**Next cell:** evaluate this derivative at the nominal UR5e configuration.

In [ ]:
def gravity_at(configuration):
    sample = mujoco.MjData(model)
    sample.qpos[:] = configuration
    mujoco.mj_forward(model, sample)  # qvel is zero in freshly allocated data.
    return sample.qfrc_bias.copy()

def gravity_jacobian(configuration, eps):
    derivative = np.zeros((model.nv, model.nv))
    for j in range(model.nv):
        tangent = np.eye(model.nv)[j]
        plus, minus = configuration.copy(), configuration.copy()
        mujoco.mj_integratePos(model, plus, tangent, eps)
        mujoco.mj_integratePos(model, minus, tangent, -eps)
        derivative[:, j] = (gravity_at(plus) - gravity_at(minus)) / (2 * eps)
    return derivative

dg_dq = gravity_jacobian(q_nominal, 1e-5)
dg_dq_half = gravity_jacobian(q_nominal, 5e-6)
np.testing.assert_allclose(dg_dq, dg_dq_half, atol=1e-6, rtol=1e-6)
print("Gravity derivative shape:", dg_dq.shape)
print("Step-halving derivative difference:", np.linalg.norm(dg_dq - dg_dq_half))

## 5. State-space representation and linearization

For stateless actuators, the continuous dynamics can be written as

$$
\dot x=
\begin{bmatrix}
W(q)v\\
M(q)^{-1}(\tau-h)
\end{bmatrix}.
$$

Here the total generalized force includes actuator, applied, passive, and constraint contributions. Activation states, if present, add their own equations.

With a timestep and integrator, MuJoCo implements a discrete map:

$$
x_{k+1}=F(x_k,u_k).
$$

Around a nominal state and input:

$$
\delta x_{k+1}\approx A\delta x_k+B\delta u_k.
$$

**What `mjd_transitionFD` returns**

- Derivatives of the **one-step discrete map**, not continuous-time Jacobians.
- State perturbation dimension: `2*nv + na`.
- Raw stored-state dimension: `nq + nv + na`.
- Quaternion perturbations use tangent coordinates.

At a non-equilibrium point, keep the nominal next state as well as the derivative matrices.

**Next cell:**

1. Load a separate Menagerie UR5e and configure direct torque motors at a contact-free nominal pose.
2. Compute the local transition matrices using central differences.
3. Predict a small perturbed step.
4. Compare the prediction with an actual `mj_step`.

Clipping, contact transitions, and changes in active constraints can invalidate a local approximation. The MuJoCo 3.13 derivative call does not support RK4.

Use an interior torque command and small perturbations to avoid saturation changes. The derivative belongs to this robot, pose, actuator configuration, and timestep; it is not a global model.

In [ ]:
linear_model = mm.load("universal_robots_ur5e")
linear_model.opt.timestep = 0.001
linear_model.opt.integrator = mujoco.mjtIntegrator.mjINT_IMPLICIT
for actuator_id in range(linear_model.nu):
    update_actuator(linear_model, actuator_id, ActuatorMotor())
linear_data = mujoco.MjData(linear_model)
linear_data.qpos[:] = q_nominal
mujoco.mj_forward(linear_model, linear_data)
u_nominal = linear_data.qfrc_bias.copy() - linear_data.qfrc_passive
linear_data.ctrl[:] = u_nominal
mujoco.mj_forward(linear_model, linear_data)
assert linear_data.ncon == 0
assert linear_model.na == 0
assert np.all(u_nominal > linear_model.actuator_ctrlrange[:, 0])
assert np.all(u_nominal < linear_model.actuator_ctrlrange[:, 1])
np.testing.assert_allclose(linear_data.qfrc_actuator, u_nominal, atol=1e-10)
nx = 2 * linear_model.nv + linear_model.na
A = np.zeros((nx, nx))
B = np.zeros((nx, linear_model.nu))
mujoco.mjd_transitionFD(linear_model, linear_data, 1e-6, True, A, B, None, None)

def one_step(dx, du):
    sample = mujoco.MjData(linear_model)
    sample.qpos[:] = q_nominal
    mujoco.mj_integratePos(linear_model, sample.qpos, dx[:linear_model.nv], 1.0)
    sample.qvel[:] = dx[linear_model.nv:]
    sample.ctrl[:] = u_nominal + du
    mujoco.mj_step(linear_model, sample)
    return sample

nominal_next = one_step(np.zeros(nx), np.zeros(linear_model.nu))
dx = np.linspace(-2e-5, 2e-5, nx)
du = np.linspace(-1e-4, 1e-4, linear_model.nu)
perturbed_next = one_step(dx, du)
delta_q_next = np.zeros(linear_model.nv)
mujoco.mj_differentiatePos(linear_model, delta_q_next, 1.0,
                          nominal_next.qpos, perturbed_next.qpos)
observed = np.r_[delta_q_next, perturbed_next.qvel - nominal_next.qvel]
predicted = A @ dx + B @ du
np.testing.assert_allclose(observed, predicted, atol=1e-8, rtol=1e-3)
print("A and B shapes:", A.shape, B.shape)
print("One-step prediction error:", np.linalg.norm(observed - predicted))

### Release graphics resources

Close the offscreen renderer when the simulation experiments are complete. Rerun the initial preview cell if you want to draw with it again.

In [ ]:
renderer.close()
renderer = None

## 6. Controlled experiments and reading

1. Compare position-servo and torque-PD inputs: which units belong in `ctrl`?
2. Vary one gain at a time and compare settling, overshoot, and final error.
3. Reduce actuator limits and distinguish requested command from delivered torque.
4. Halve the finite-difference step and inspect gravity-derivative changes.
5. Increase the state/input perturbation and measure the linear model's prediction error.
6. Change the timestep and recompute the discrete-time matrices. Why must they change?

**Documentation**

- [Actuation and dynamics](https://mujoco.readthedocs.io/en/stable/computation/index.html).
- [Discrete transition derivatives](https://mujoco.readthedocs.io/en/stable/APIreference/APIfunctions.html#mjd-transitionfd).
- [Actuator XML specification](https://mujoco.readthedocs.io/en/stable/XMLreference.html#actuator).
- [Model editing with MjSpec](https://mujoco.readthedocs.io/en/stable/python.html#model-editing).

Use the course's pinned MuJoCo version when consulting API signatures.